# Tối ưu hoá GHI dữ liệu trong Spark — Notebook thực hành

Notebook đồng hành với phần lý thuyết về tối ưu ghi dữ liệu. Mục tiêu: **thấy bằng con số** sự khác biệt giữa các kỹ thuật ghi.

## Nội dung

1. **Setup** — SparkSession và dữ liệu mẫu
2. **So sánh format**: CSV, JSON, Parquet, ORC — kích thước, thời gian ghi, thời gian đọc
3. **coalesce vs repartition** — đo thời gian + số file + phân bố data
4. **partitionBy** — thực hành phân vùng đúng vs sai
5. **Small File Problem** — tái hiện vấn đề + 3 cách khắc phục
6. **Pattern best practice** — `repartition() + partitionBy()` kết hợp
7. **Bonus**: compression options (snappy/gzip/zstd)

## Yêu cầu
```bash
pip install pyspark==3.5.0
```
Chạy được trên máy local, không cần cluster.

---
## 1. Setup

In [1]:
import os
import time
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("WriteOptimization")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

BASE = "file:///opt/workspace/data/demo-module-4.4/write"

if os.path.exists(BASE):
    shutil.rmtree(BASE)
os.makedirs(BASE, exist_ok=True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/15 13:45:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8


In [2]:
# Helpers chung
def dir_size_mb(path):
    """Tổng dung lượng thư mục (MB)"""
    total = 0
    for dirpath, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / 1024 / 1024

def count_files(path, ext=None):
    """Đếm số file (có thể lọc theo extension)"""
    n = 0
    for _, _, files in os.walk(path):
        for f in files:
            if ext is None or f.endswith(ext):
                n += 1
    return n

def timed(label, func):
    """Đo thời gian thực thi"""
    t0 = time.time()
    result = func()
    dt = time.time() - t0
    print(f"  [{label:30s}] {dt:7.3f}s")
    return dt, result

In [3]:
# Dữ liệu mẫu: 2 triệu dòng, mô phỏng bảng sales
# Có cố ý tạo skew: HCM chiếm 50% data
N_ROWS = 2000000

df = (spark.range(0, N_ROWS)
      .withColumn("year",     (F.col("id") % 3 + 2023).cast("int"))
      .withColumn("month",    (F.col("id") % 12 + 1).cast("int"))
      # Tạo skew có chủ ý: 50% HCM, 25% HN, còn lại chia đều
      .withColumn("city", F.when(F.col("id") % 4 < 2, F.lit("HCM"))
                          .when(F.col("id") % 4 == 2, F.lit("HN"))
                          .when(F.col("id") % 8 == 3, F.lit("DN"))
                          .when(F.col("id") % 8 == 7, F.lit("CT"))
                          .otherwise(F.lit("HP")))
      .withColumn("user_id", F.col("id").cast("long"))  # high-cardinality cho demo
      .withColumn("amount",  (F.rand() * 1000).cast("double"))
      .withColumn("category", F.concat(F.lit("cat_"), (F.col("id") % 20).cast("string"))))

# Cache để các phép đo sau không tốn thời gian sinh lại data
df.cache()
print(f"Tổng số dòng: {df.count():,}")
print(f"Số partition ban đầu: {df.rdd.getNumPartitions()}")

print("\nPhân bố city (có skew):")
df.groupBy("city").count().orderBy(F.desc("count")).show()

Tổng số dòng: 2,000,000
Số partition ban đầu: 32

Phân bố city (có skew):


+----+-------+
|city|  count|
+----+-------+
| HCM|1000000|
|  HN| 500000|
|  CT| 250000|
|  DN| 250000|
+----+-------+



In [4]:
path_csv     = BASE + "/format_csv"
path_json     = BASE + "/format_json"
path_parquet  = BASE + "/format_parquet"
path_orc     = BASE + "/format_orc"

---
## 2. So sánh các định dạng file

**Mục tiêu**: đo trực tiếp 4 con số cho mỗi format — kích thước, thời gian ghi, thời gian đọc full, thời gian đọc với filter.

In [5]:
results = {}  # lưu kết quả để tổng kết

# CSV
print("=== CSV ===")
t_write, _ = timed("write", lambda: df.write.mode("overwrite").option("header", "true").csv(path_csv))
t_read, _  = timed("read full count", lambda: spark.read.option("header", "true").csv(path_csv).count())
t_filter, _ = timed("read with filter", lambda: 
    spark.read.option("header", "true").csv(path_csv)
         .filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
results["CSV"] = {"size_mb": dir_size_mb(path_csv), "write": t_write, "read": t_read, "filter": t_filter}

=== CSV ===


  [write                         ]   9.531s


  [read full count               ]   2.979s


  [read with filter              ]   2.920s


In [6]:
# JSON
print("=== JSON ===")
t_write, _ = timed("write", lambda: df.write.mode("overwrite").json(path_json))
t_read, _  = timed("read full count", lambda: spark.read.json(path_json).count())
t_filter, _ = timed("read with filter", lambda: 
    spark.read.json(path_json)
         .filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
results["JSON"] = {"size_mb": dir_size_mb(path_json), "write": t_write, "read": t_read, "filter": t_filter}

=== JSON ===


  [write                         ]   8.164s


  [read full count               ]   4.646s


  [read with filter              ]   3.957s


In [7]:
# Parquet
print("=== Parquet ===")
t_write, _ = timed("write", lambda: df.write.mode("overwrite").parquet(path_parquet))
t_read, _  = timed("read full count", lambda: spark.read.parquet(path_parquet).count())
t_filter, _ = timed("read with filter", lambda: 
    spark.read.parquet(path_parquet)
         .filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
results["Parquet"] = {"size_mb": dir_size_mb(path_parquet), "write": t_write, "read": t_read, "filter": t_filter}

=== Parquet ===


26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:00 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 24.52% for 31 writers
26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 25.33% for 30 writers
26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 26.21% for 29 writers
26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 27.14% for 28 writers
26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 28.15% for 27 writers
26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 29.23% for 26 writers
26/05/15 13:46:02 WARN MemoryManager: Total allocation exceeds 9

  [write                         ]   8.602s


  [read full count               ]   1.699s


  [read with filter              ]   1.871s


In [8]:
# ORC
print("=== ORC ===")
t_write, _ = timed("write", lambda: df.write.mode("overwrite").orc(path_orc))
t_read, _  = timed("read full count", lambda: spark.read.orc(path_orc).count())
t_filter, _ = timed("read with filter", lambda: 
    spark.read.orc(path_orc)
         .filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
results["ORC"] = {"size_mb": dir_size_mb(path_orc), "write": t_write, "read": t_read, "filter": t_filter}

=== ORC ===


  [write                         ]   7.576s


  [read full count               ]   1.666s


  [read with filter              ]   2.087s


In [9]:
# Tổng kết — in bảng so sánh
print(f"{'Format':<10} {'Size (MB)':<12} {'Write (s)':<12} {'Read (s)':<12} {'Filter (s)':<12}")
print("-" * 60)
for fmt, r in results.items():
    print(f"{fmt:<10} {r['size_mb']:<12.4f} {r['write']:<12.3f} {r['read']:<12.3f} {r['filter']:<12.3f}")

# print(f"\n→ Parquet nhỏ hơn CSV {results['CSV']['size_mb']/results['Parquet']['size_mb']:.1f}x")
# print(f"→ Parquet đọc nhanh hơn CSV {results['CSV']['read']/results['Parquet']['read']:.1f}x")
# print(f"→ Parquet filter nhanh hơn CSV {results['CSV']['filter']/results['Parquet']['filter']:.1f}x")

Format     Size (MB)    Write (s)    Read (s)     Filter (s)  
------------------------------------------------------------
CSV        0.0000       9.531        2.979        2.920       
JSON       0.0000       8.164        4.646        3.957       
Parquet    0.0000       8.602        1.699        1.871       
ORC        0.0000       7.576        1.666        2.087       


**Quan sát**:
- Parquet/ORC nhỏ hơn 5-10x so với CSV/JSON nhờ columnar compression.
- Filter trên Parquet/ORC nhanh hơn nhiều nhờ predicate pushdown.
- Write của Parquet có thể chậm hơn CSV một chút (do nén), nhưng đó là chi phí một lần đáng giá.

---
## 3. coalesce vs repartition

**Mục tiêu**: hiểu rõ khi nào dùng cái nào qua đo lường thời gian + số file + phân bố.

In [10]:

# Trạng thái ban đầu: bao nhiêu partition?
n_default = df.rdd.getNumPartitions()
print(f"Số partition mặc định: {n_default}")

# Tăng partition lên 50 để demo rõ hơn
df_50 = df.repartition(50)
print(f"Sau repartition(50): {df_50.rdd.getNumPartitions()} partition")

Số partition mặc định: 32


Sau repartition(50): 50 partition


In [11]:
# Test 1: ghi với 50 partition mặc định (không tối ưu)
path_50 = f"{BASE}/coalesce_50_default"
print("Ghi với 50 partition (không tối ưu):")
t_50, _ = timed("50 partition", lambda: df_50.write.mode("overwrite").parquet(path_50))
print(f"  → {count_files(path_50, '.parquet')} file parquet, {dir_size_mb(path_50):.2f} MB")

Ghi với 50 partition (không tối ưu):


26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 24.52% for 31 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 25.33% for 30 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 26.21% for 29 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 27.14% for 28 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 28.15% for 27 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 29.23% for 26 writers
26/05/15 13:46:25 WARN MemoryManager: Total allocation exceeds 9

26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:26 WARN MemoryManager: Total allocation exceeds 95

  [50 partition                  ]  10.740s
  → 0 file parquet, 0.00 MB


In [12]:
# Test 2: coalesce(5) — gộp xuống 5 partition, KHÔNG shuffle
path_coal = f"{BASE}/coalesce_5"
print("coalesce(5) — không shuffle:")
t_coal, _ = timed("coalesce(5)", lambda: df_50.coalesce(5).write.mode("overwrite").parquet(path_coal))
print(f"  → {count_files(path_coal, '.parquet')} file parquet, {dir_size_mb(path_coal):.2f} MB")

coalesce(5) — không shuffle:


  [coalesce(5)                   ]   3.547s
  → 0 file parquet, 0.00 MB


In [13]:
# Test 3: repartition(5) — shuffle để chia đều
path_rep = f"{BASE}/repartition_5"
print("repartition(5) — có shuffle, phân bố đều:")
t_rep, _ = timed("repartition(5)", lambda: df_50.repartition(5).write.mode("overwrite").parquet(path_rep))
print(f"  → {count_files(path_rep, '.parquet')} file parquet, {dir_size_mb(path_rep):.2f} MB")

repartition(5) — có shuffle, phân bố đều:


  [repartition(5)                ]   2.820s
  → 0 file parquet, 0.00 MB


In [14]:
# So sánh tổng quan
print(f"{'Method':<25} {'Time (s)':<12} {'Files':<10} {'Size (MB)':<10}")
print("-" * 60)
print(f"{'50 partition (default)':<25} {t_50:<12.3f} {count_files(path_50, '.parquet'):<10} {dir_size_mb(path_50):<10.2f}")
print(f"{'coalesce(5)':<25} {t_coal:<12.3f} {count_files(path_coal, '.parquet'):<10} {dir_size_mb(path_coal):<10.2f}")
print(f"{'repartition(5)':<25} {t_rep:<12.3f} {count_files(path_rep, '.parquet'):<10} {dir_size_mb(path_rep):<10.2f}")

print(f"\n→ coalesce nhanh hơn repartition {t_rep/t_coal:.2f}x (không shuffle)")

Method                    Time (s)     Files      Size (MB) 
------------------------------------------------------------
50 partition (default)    10.740       0          0.00      
coalesce(5)               3.547        0          0.00      
repartition(5)            2.820        0          0.00      

→ coalesce nhanh hơn repartition 0.80x (không shuffle)


In [15]:
# So sánh phân bố data trong các file output
def file_size_distribution(path):
    sizes = []
    for root, _, files in os.walk(path):
        for f in files:
            if f.endswith(".parquet"):
                sizes.append(os.path.getsize(os.path.join(root, f)) / 1024 / 1024)
    return sorted(sizes, reverse=True)

print("Phân bố kích thước file (MB) — chú ý độ ĐỀU:")
print(f"\ncoalesce(5):    {[f'{s:.2f}' for s in file_size_distribution(path_coal)]}")
print(f"repartition(5): {[f'{s:.2f}' for s in file_size_distribution(path_rep)]}")

print("\n💡 Quan sát: repartition cho file size ĐỀU NHAU; coalesce có thể có file lớn hơn các file khác.")

Phân bố kích thước file (MB) — chú ý độ ĐỀU:

coalesce(5):    []
repartition(5): []

💡 Quan sát: repartition cho file size ĐỀU NHAU; coalesce có thể có file lớn hơn các file khác.


**Bài học**:
- `coalesce` **nhanh hơn** vì không shuffle, nhưng có thể phân bố không đều.
- `repartition` **chậm hơn** (shuffle), nhưng phân bố đều — chống data skew.
- Quy tắc: dùng `coalesce` khi giảm partition nhẹ; dùng `repartition` khi cần đảm bảo đều hoặc tăng partition.

---
## 4. partitionBy — Phân vùng đúng vs sai

In [16]:
# ✅ ĐÚNG: partition theo year + month (cardinality vừa)
path_good = f"{BASE}/partition_good"
t_good, _ = timed("partitionBy(year, month)", 
    lambda: df.write.mode("overwrite").partitionBy("year", "month").parquet(path_good))
n_good_files = count_files(path_good, ".parquet")
size_good = dir_size_mb(path_good)
print(f"  → {n_good_files} file, {size_good:.2f} MB, TB/file: {size_good*1024/n_good_files:.1f} KB")

26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:46:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 24.52% for 31 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 25.33% for 30 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 26.21% for 29 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 27.14% for 28 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 28.15% for 27 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 29.23% for 26 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 9

26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:51 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:46:52 WARN MemoryManager: Total allocation exceeds 95.

...

26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00%

26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:46:56 WARN MemoryManager: Total allocation exceeds 95.0

  [partitionBy(year, month)      ] 1162.830s


ZeroDivisionError: float division by zero

In [17]:
# ❌ SAI: partition theo user_id (high cardinality)
# CHÚ Ý: limit lại để tránh chạy quá lâu — user_id là long unique
# Ta sẽ partition theo category (20 giá trị) trên 100K dòng để mô phỏng
df_small = df.limit(100_000)  # giảm xuống để demo nhanh

# Test với category (20 giá trị) — vừa phải
path_cat = f"{BASE}/partition_category"
t_cat, _ = timed("partitionBy(category) - OK", 
    lambda: df_small.write.mode("overwrite").partitionBy("category").parquet(path_cat))
n_cat_files = count_files(path_cat, ".parquet")
size_cat = dir_size_mb(path_cat)
print(f"  → {n_cat_files} file, {size_cat:.2f} MB, TB/file: {size_cat*1024/n_cat_files:.1f} KB")

  [partitionBy(category) - OK    ]   7.620s


ZeroDivisionError: float division by zero

In [18]:
# ❌ SAI THẢM HOẠ: partition theo cột high-cardinality
# Mô phỏng: tạo cột có 1000 giá trị distinct
df_high_card = df_small.withColumn("bad_col", (F.col("id") % 1000).cast("int"))

path_bad = f"{BASE}/partition_bad"
t_bad, _ = timed("partitionBy(bad_col) - BAD", 
    lambda: df_high_card.write.mode("overwrite").partitionBy("bad_col").parquet(path_bad))
n_bad_files = count_files(path_bad, ".parquet")
size_bad = dir_size_mb(path_bad)
print(f"  → {n_bad_files} file, {size_bad:.2f} MB, TB/file: {size_bad*1024/n_bad_files:.2f} KB")
print(f"\n⚠️  TB/file CHỈ {size_bad*1024/n_bad_files:.2f} KB — quá nhỏ! Đây là small file problem.")

  [partitionBy(bad_col) - BAD    ] 233.110s


ZeroDivisionError: float division by zero

In [19]:
# Tổng kết partition
print(f"{'Strategy':<35} {'Files':<10} {'KB/file':<12} {'Verdict'}")
print("-" * 75)
print(f"{'partitionBy(year, month) — 2M rows':<35} {n_good_files:<10} {size_good*1024/n_good_files:<12.1f} ✅ OK")
print(f"{'partitionBy(category) — 100K rows':<35} {n_cat_files:<10} {size_cat*1024/n_cat_files:<12.1f} ⚠️  borderline")
print(f"{'partitionBy(bad_col 1000 distinct)':<35} {n_bad_files:<10} {size_bad*1024/n_bad_files:<12.1f} ❌ thảm hoạ")

Strategy                            Files      KB/file      Verdict
---------------------------------------------------------------------------


ZeroDivisionError: float division by zero

**Bài học**: cardinality của cột partition phải phù hợp với lượng data. Quy tắc thô: `total_size / num_partitions ≥ 100MB`.

---
## 5. Small File Problem — Tái hiện + 3 cách khắc phục

In [20]:
# Tái hiện: ghi với quá nhiều partition
df_skewed = df.repartition(200)  # 200 partition trên 2M rows = file nhỏ

path_smf = f"{BASE}/small_files"
t_smf, _ = timed("Ghi với 200 partition (SAI)", 
    lambda: df_skewed.write.mode("overwrite").parquet(path_smf))
n_smf = count_files(path_smf, ".parquet")
size_smf = dir_size_mb(path_smf)
print(f"  → {n_smf} file, {size_smf:.2f} MB, TB/file: {size_smf*1024/n_smf:.1f} KB ❌")

26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 25.33% for 30 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 26.21% for 29 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 27.14% for 28 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 28.15% for 27 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 29.23% for 26 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 30.40% for 25 writers
26/05/15 14:10:07 WARN MemoryManager: Total allocation exceeds 9

26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00%

26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:09 WARN MemoryManager: Total allocation exceeds 95.00% 

26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:10:10 WARN MemoryManager: Total allocation exceeds 95.00%

  [Ghi với 200 partition (SAI)   ]  39.550s


ZeroDivisionError: float division by zero

In [21]:
# Hậu quả: thử đọc các file nhỏ này
print("Thử đọc thư mục có nhiều file nhỏ:")
t_read_smf, _ = timed("Đọc 200 file nhỏ", 
    lambda: spark.read.parquet(path_smf).count())

# So sánh: ghi đúng (coalesce) + đọc
path_compact = f"{BASE}/compacted"
df.coalesce(4).write.mode("overwrite").parquet(path_compact)
n_compact = count_files(path_compact, ".parquet")
size_compact = dir_size_mb(path_compact)
print(f"\nFile compact: {n_compact} file, TB/file: {size_compact*1024/n_compact:.1f} KB")

t_read_compact, _ = timed("Đọc 4 file lớn", 
    lambda: spark.read.parquet(path_compact).count())

print(f"\n→ Đọc file đã compact nhanh hơn {t_read_smf/t_read_compact:.1f}x")

Thử đọc thư mục có nhiều file nhỏ:


  [Đọc 200 file nhỏ              ]   8.333s


ZeroDivisionError: float division by zero

In [22]:
# Giải pháp 3: Compaction job — gộp file nhỏ đã có sẵn
print("Giải pháp 3: compaction job cho data đã ghi sai")

path_after_compact = f"{BASE}/after_compact"
t_compact, _ = timed("Compact 200 → 4 file", 
    lambda: spark.read.parquet(path_smf).coalesce(4).write.mode("overwrite").parquet(path_after_compact))

print(f"  → Trước: {n_smf} file ({size_smf*1024/n_smf:.1f} KB/file)")
print(f"  → Sau:   {count_files(path_after_compact, '.parquet')} file ({dir_size_mb(path_after_compact)*1024/count_files(path_after_compact, '.parquet'):.1f} KB/file)")
print("\n💡 Compaction job nên chạy định kỳ (hàng đêm/tuần) trên data lake.")

Giải pháp 3: compaction job cho data đã ghi sai


  [Compact 200 → 4 file          ]  13.435s


ZeroDivisionError: float division by zero

---
## 6. Pattern best practice: repartition() + partitionBy()

**Mục tiêu**: thấy được tại sao kết hợp 2 cái này quan trọng.

In [23]:
# ❌ KHÔNG repartition trước partitionBy — tạo nhiều file/thư mục
path_naive = f"{BASE}/partition_naive"
df_50 = df.repartition(50)  # 50 partition
df_50.write.mode("overwrite").partitionBy("city").parquet(path_naive)
n_naive = count_files(path_naive, ".parquet")

# Đếm file trong mỗi thư mục city
print("❌ KHÔNG repartition trước:")
for city_dir in sorted(os.listdir(path_naive)):
    if city_dir.startswith("city="):
        n = count_files(os.path.join(path_naive, city_dir), ".parquet")
        print(f"  {city_dir}: {n} file")
print(f"  TỔNG: {n_naive} file")

26/05/15 14:11:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:11:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 29.23% for 26 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 30.40% for 25 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 31.67% for 24 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 33.04% for 23 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 34.55% for 22 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 36.19% for 21 writers
26/05/15 14:11:10 WARN MemoryManager: Total allocation exceeds 9

26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 14:11:11 WARN MemoryManager: Total allocation exceeds 95.

...

26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95

26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 14:11:12 WARN MemoryManager: Total allocation exceeds 95.00

❌ KHÔNG repartition trước:


FileNotFoundError: [Errno 2] No such file or directory: 'file:///opt/workspace/data/demo-module-4.4/write/partition_naive'

In [24]:
# ✅ CÓ repartition theo cùng cột với partitionBy
path_smart = f"{BASE}/partition_smart"
(df.repartition("city")  # shuffle để gom cùng city vào 1 Spark partition
   .write.mode("overwrite")
   .partitionBy("city")
   .parquet(path_smart))
n_smart = count_files(path_smart, ".parquet")

print("\n✅ CÓ repartition('city') trước:")
for city_dir in sorted(os.listdir(path_smart)):
    if city_dir.startswith("city="):
        n = count_files(os.path.join(path_smart, city_dir), ".parquet")
        print(f"  {city_dir}: {n} file")
print(f"  TỔNG: {n_smart} file")

print(f"\n→ Giảm từ {n_naive} → {n_smart} file output ({n_naive/n_smart:.1f}x ít hơn)")


✅ CÓ repartition('city') trước:


FileNotFoundError: [Errno 2] No such file or directory: 'file:///opt/workspace/data/demo-module-4.4/write/partition_smart'

**Bài học**: khi dùng `partitionBy`, hãy `repartition` theo CHÍNH cột partition trước → mỗi thư mục output chỉ có 1 file thay vì N file nhỏ.

---
## 7. So sánh compression options

In [25]:
# Test các loại compression của Parquet
compressions = ["snappy", "gzip", "zstd", "none"]
comp_results = {}

for comp in compressions:
    path_comp = f"{BASE}/comp_{comp}"
    print(f"\n=== Compression: {comp} ===")
    t_w, _ = timed("write", lambda c=comp: 
        df.coalesce(4).write.mode("overwrite")
          .option("compression", c).parquet(path_comp))
    t_r, _ = timed("read", lambda: spark.read.parquet(path_comp).count())
    size = dir_size_mb(path_comp)
    comp_results[comp] = {"size": size, "write": t_w, "read": t_r}

print(f"\n{'Compression':<12} {'Size (MB)':<12} {'Write (s)':<12} {'Read (s)':<12}")
print("-" * 50)
for comp, r in comp_results.items():
    print(f"{comp:<12} {r['size']:<12.2f} {r['write']:<12.3f} {r['read']:<12.3f}")


=== Compression: snappy ===


  [write                         ]   2.298s


  [read                          ]   0.368s

=== Compression: gzip ===


  [write                         ]   2.486s


  [read                          ]   0.384s

=== Compression: zstd ===


  [write                         ]   1.949s


  [read                          ]   0.401s

=== Compression: none ===


  [write                         ]   2.771s


  [read                          ]   0.391s

Compression  Size (MB)    Write (s)    Read (s)    
--------------------------------------------------
snappy       0.00         2.298        0.368       
gzip         0.00         2.486        0.384       
zstd         0.00         1.949        0.401       
none         0.00         2.771        0.391       


**Quan sát**:
- `snappy`: cân bằng tốt nhất, default của Spark — tốc độ + nén OK.
- `gzip`: nén tốt hơn, nhưng CPU cao, ghi chậm hơn.
- `zstd`: cân bằng tốt nhất hiện tại — nén tốt như gzip, nhanh như snappy.
- `none`: không nén, file lớn, không khuyến nghị.

Khuyến nghị: dùng **zstd** cho data lưu lâu dài (cold storage), **snappy** cho data truy cập thường xuyên.

---
## 8. Tổng kết

### Checklist ghi dữ liệu trong Spark

| ✅ Nên làm | ❌ Tránh |
|----------|----------|
| Dùng Parquet/ORC | Dùng CSV cho data lake |
| `coalesce()` để giảm file nhẹ | `repartition(10000)` |
| `partitionBy` cột low-cardinality (year, month, region) | `partitionBy(user_id)` |
| Mỗi file 128MB - 1GB | File < 10MB |
| `repartition(col)` trước `partitionBy(col)` | `partitionBy` mà không repartition trước |
| Compression: snappy hoặc zstd | Không nén |
| Compaction job định kỳ | Để small files tích luỹ |



### Bài tập đề xuất

1. Tăng `N_ROWS` lên 10 triệu, partition theo `(year, month, city)` — xem có còn small file không?
2. Tạo data có skew cực mạnh (1 city chiếm 90%) — so sánh `repartition("city")` vs `repartition(10, "city")`.
3. Implement 1 compaction job hoàn chỉnh: scan thư mục có file < 10MB, gộp lại.
4. Đo write performance khi `partitionBy(year, month, day)` so với chỉ `partitionBy(year, month)`.

In [26]:
# Cleanup
df.unpersist()
spark.stop()
print("Done!")

Done!
